In [ ]:
# revT2V — Kaggle launcher. Runs ONE step (STEP) via Save & Run All (Commit).
# Order: rank_check -> smoke_test -> train_conv_lora -> train_attn_injection (see KAGGLE.md).
# Needs: GPU on, Internet on, Secret HF_TOKEN attached, revt2v-data-2 (the
# dataset2/ shards) added as input. revt2v-data (dataset1/, the old shards)
# should NOT be attached — training uses dataset2 only; see configs/*.yaml.
# Settings
STEP = "rank_check"  # rank_check | smoke_test | smoke_test_attn_injection | train_conv_lora | train_attn_injection

HF_DATASET_REPO = "silentlooop/revt2v-data"  # dataset2/ shards (fallback if no Kaggle input)
HF_CHECKPOINT_REPO = "silentlooop/revt2v-ckpt"  # runs go to conv_lora/ and attn_injection/, never the root latest.pt
REPO_URL = "https://github.com/silentlooop/revT2V.git"
REPO_DIR = "/kaggle/working/revT2V"

STEPS = ("rank_check", "smoke_test", "smoke_test_attn_injection", "train_conv_lora", "train_attn_injection")
assert STEP in STEPS, f"STEP must be one of {STEPS}, got {STEP!r}"
print("STEP =", STEP)

In [ ]:
# 1. Clone/pull the repo and install it
print("Cloning/pulling repo...", flush=True)
!git clone -q {REPO_URL} {REPO_DIR} 2>/dev/null || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

print("Installing revt2v + deps (pip -q is silent here; can take a few minutes)...", flush=True)
!pip install -q -e .
print("pip install done", flush=True)

# Kaggle preinstalls torchao 0.10.0, which peft rejects outright (it requires >=0.16.0
# for its optional torchao LoRA dispatcher) — we never use torchao-quantized layers,
# so removing it avoids the crash without touching torch/CUDA-adjacent packages.
!pip uninstall -y torchao -q
print("Cell 1 done", flush=True)

In [ ]:
# 2. HF token from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Kaggle Secret HF_TOKEN is empty / not attached to this notebook"

from huggingface_hub import whoami
print("HF user:", whoami(token=os.environ["HF_TOKEN"])["name"])

In [ ]:
# 3. GPU check (rank_check runs on CPU, every other step needs a GPU)
import peft
import torch

print("torch", torch.__version__, "| peft", peft.__version__)
from peft.tuners.lora import layer as lora_layer
assert hasattr(lora_layer, "Conv3d"), "this peft has no Conv3d LoRA; conv_lora needs peft>=0.21"

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
else:
    assert STEP == "rank_check", "No GPU visible — Settings > Accelerator > GPU"
    print("No GPU (fine for rank_check)")

In [ ]:
# 4. Helpers
import math
import re
import shutil
import subprocess
import sys


def run(*args):
    """Run a command, stream its output live, fail the notebook on a non-zero exit, return the output."""
    cmd = [str(a) for a in args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return "".join(lines)


def train(method, *extra):
    return run(sys.executable, "scripts/train.py", "--config", f"configs/{method}.yaml",
               "--hf-repo", HF_CHECKPOINT_REPO, "--hf-dataset-repo", HF_DATASET_REPO, *extra)

In [ ]:
# 4b. Dataset from the Kaggle input: link dataset2's shards (+ manifest) into
# data/latents (no HF download). dataset2 is identified by its manifest.jsonl
# — the original dataset1 shards have none — so this works even if both
# revt2v-data and revt2v-data-2 end up attached as inputs.
import shutil
from pathlib import Path

DATA_ROOT = Path(REPO_DIR) / "data/latents"
INPUT_ROOT = Path("/kaggle/input")

inputs = sorted(str(p) for p in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
print("Kaggle inputs:", inputs or "none")

manifests = sorted(INPUT_ROOT.rglob("manifest.jsonl")) if INPUT_ROOT.exists() else []
if len(manifests) > 1:
    raise RuntimeError(f"Multiple manifest.jsonl found under Kaggle inputs, expected one (dataset2): {manifests}")

if manifests:
    src_dir = manifests[0].parent
    shards = sorted(src_dir.glob("*.pt"))
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for shard in shards:
        link = DATA_ROOT / shard.name
        if not link.exists():
            link.symlink_to(shard)
    manifest_dst = DATA_ROOT / "manifest.jsonl"
    if not manifest_dst.exists():
        shutil.copy(manifests[0], manifest_dst)
    print(f"{len(shards)} shards + manifest from {src_dir} linked into {DATA_ROOT}")
else:
    print(f"No dataset2 manifest in the Kaggle inputs; train.py downloads it from {HF_DATASET_REPO}/dataset2")

In [ ]:
# 5. Run the selected step
def run_smoke_test(method):
    """20 steps, saving (and pushing) every 10, into its own Hub folder; then resume from it."""
    from huggingface_hub import hf_hub_download
    from revt2v.utils import list_hub_files

    smoke_folder = f"smoke_test/{method}"
    smoke_ckpt_dir = f"results/checkpoints/smoke_test/{method}"
    smoke_args = ["--hf-subfolder", smoke_folder, "--checkpoint-dir", smoke_ckpt_dir,
                  "--save-every-steps", 10, "--push-every-seconds", 0, "--log-every-steps", 1,
                  "--set", "grad_accum_steps=2", "sample_every_steps=20"]
    shutil.rmtree(smoke_ckpt_dir, ignore_errors=True)
    out = train(method, "--fresh", "--max-steps", 20, *smoke_args)

    print("\n" + "=" * 70 + "\nSMOKE TEST REPORT")
    print(re.search(r"LoRA-wrapped layers: .*", out).group(0))
    print(re.search(r"Dataset: .*", out).group(0))
    losses = [float(x) for x in re.findall(r"step=\d+ loss=(\S+)", out)]
    print(f"losses ({len(losses)} steps):", [round(l, 4) for l in losses])
    assert len(losses) == 20 and all(map(math.isfinite, losses)), "non-finite or missing losses"
    assert "Non-finite loss" not in out, "a micro-batch had a non-finite loss"
    print("all losses finite: OK")

    step_times = sorted(float(x) for x in re.findall(r"step=\d+ loss=\S+ step_time=([\d.]+)s", out)[1:])
    sec_per_micro = step_times[len(step_times) // 2] / 2  # smoke uses grad_accum 2
    print(f"~{sec_per_micro:.1f} s per micro-batch -> full run (1000 steps x grad_accum 8) ~ {sec_per_micro * 8000 / 3600:.1f} h")

    hub_ckpt = f"{smoke_folder}/latest.pt"
    assert hub_ckpt in list_hub_files(HF_CHECKPOINT_REPO, repo_type="model"), f"{hub_ckpt} not on the Hub"
    pushed = torch.load(hf_hub_download(HF_CHECKPOINT_REPO, hub_ckpt, force_download=True), map_location="cpu")
    print(f"checkpoint uploaded: {HF_CHECKPOINT_REPO}/{hub_ckpt} (step {pushed['step']}, "
          f"{len(pushed['model'])} LoRA tensors, method {pushed['metadata']['method']})")
    assert pushed["step"] == 19

    # Resume: wipe the local copy so it has to come back from the Hub, then run 2 more steps.
    shutil.rmtree(smoke_ckpt_dir)
    out = train(method, "--max-steps", 22, *smoke_args)
    assert "Resumed from step 19" in out, "resume did not pick up the Hub checkpoint"
    resumed = [float(x) for x in re.findall(r"step=(?:20|21) loss=(\S+)", out)]
    assert len(resumed) == 2 and all(map(math.isfinite, resumed)), resumed
    print("resume from Hub checkpoint: OK (step 19 -> 21, losses", [round(l, 4) for l in resumed], ")")
    print(f"SMOKE TEST PASSED ({method})")


if STEP == "rank_check":
    out = run(sys.executable, "scripts/conv_rank_check.py")
    open("/kaggle/working/rank_check.txt", "w").write(out)

elif STEP == "train_conv_lora":
    train("conv_lora")

elif STEP == "train_attn_injection":
    train("attn_injection")

elif STEP == "smoke_test":
    run_smoke_test("conv_lora")

elif STEP == "smoke_test_attn_injection":
    run_smoke_test("attn_injection")

In [ ]:
# 6. Outputs (checkpoints and samples are also on the HF Hub)
!ls -la /kaggle/working
!find results -name "*.pt" -o -name "*.mp4" | head -50